# Fase CRISP-DM: Carga, Exploración Informática e Imputación Inteligente
**Plataforma**: AgroData Intelligence Platform (AgroStatsApp)  
**Propósito**: Este notebook carga los 10 conjuntos de datos agropecuarios desde `data/RAW`, ejecuta un perfilamiento informático exhaustivo (nulos, duplicados, metadatos, granularidad) y aplica el motor de imputación inteligente multivariada bajo el marco de Rubin.

In [4]:
# 1. Configuración de Entorno Resiliente (Google Colab / VS Code / Jupyter Local)
import os
import sys
import subprocess
from pathlib import Path

def setup_environment():
    # A. Detección y preparación automática para Google Colab
    if 'google.colab' in sys.modules or Path('/content').exists():
        print('[INFO] Entorno detectado: Google Colab.')
        repo_dir = Path('/content/Statsfirm')
        if not repo_dir.exists():
            print('[INFO] Clonando repositorio oficial Statsfirm en Colab...')
            subprocess.run(['git', 'clone', 'https://github.com/adansanchezc1-spec/Statsfirm.git', '/content/Statsfirm'], check=True)
        else:
            print('[INFO] Actualizando repositorio en Colab...')
            subprocess.run(['git', '-C', '/content/Statsfirm', 'pull'], check=False)
        
        app_dir = repo_dir / 'AgroStats AndTech' / 'AgroStatsApp'
        if app_dir.exists():
            os.chdir(str(app_dir))
            src_dir = app_dir / 'src'
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            print(f'[OK] Directorio de trabajo establecido en: {app_dir}')
            print(f'[OK] Carpeta src agregada a sys.path: {src_dir}')
            return

    # B. Detección dinámica en Entorno Local (Windows / Linux / WSL / VS Code)
    candidates = [
        Path.cwd() / 'src',
        Path.cwd() / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path.cwd().parent / 'src',
        Path.cwd().parent.parent / 'src',
        Path.cwd().parent.parent / 'AgroStats AndTech' / 'AgroStatsApp' / 'src',
        Path(r'c:\Users\ADAN\OneDrive\Documentos\Statsfirm\AgroStats AndTech\AgroStatsApp\src'),
        Path('/mnt/c/Users/ADAN/OneDrive/Documentos/Statsfirm/AgroStats AndTech/AgroStatsApp/src'),
    ]
    
    curr = Path.cwd().resolve()
    for _ in range(6):
        target = curr / 'AgroStats AndTech' / 'AgroStatsApp' / 'src'
        if target.exists() and (target / 'notebook_code').is_dir():
            candidates.insert(0, target)
            break
        curr = curr.parent

    for c in candidates:
        if c.exists() and (c / 'notebook_code').is_dir():
            resolved = str(c.resolve())
            if resolved not in sys.path:
                sys.path.insert(0, resolved)
            print(f'[OK] Módulo src localizado localmente en: {resolved}')
            return

setup_environment()

# Importación de utilidades informáticas y de perfilamiento
from notebook_code import (
    RawDataLoader,
    DatasetProfiler,
    NotebookImputerBridge,
    load_all_raw_datasets,
    load_raw_dataset,
    explore_dataset,
)

print('[OK] Módulos de notebook_code e imputer importados exitosamente.')


ModuleNotFoundError: No module named 'notebook_code'

In [ ]:
# 2. Carga Automatizada de los 10 Datasets desde data/RAW
datasets = load_all_raw_datasets()

print(f'Total de datasets cargados desde data/RAW: {len(datasets)}\n')
for name, df in datasets.items():
    print(f'• {name:25} -> {df.shape[0]:5} filas x {df.shape[1]:2} columnas')

In [ ]:
# 3. Selección y Exploración Informática de un Dataset (ej. Pluviometría IDEAM)
df_pluvio = datasets['ideam_pluvio']

# Diagnóstico informático en formato estructurado
perfil = explore_dataset(df_pluvio, dataset_name='IDEAM Pluviometría')

print('=== METADATOS Y CONTEXTO ===')
for k, v in perfil['metadata'].items():
    print(f'  {k}: {v}')

print('\n=== ANÁLISIS DE DATOS NULOS Y AUSENCIAS ===')
print(f"  Tasa global de nulos: {perfil['missingness']['global_missing_rate_pct']} %")
print(f"  Filas completas: {perfil['missingness']['complete_rows_count']} ({perfil['missingness']['complete_rows_pct']} %)")
print(f"  Columnas con nulos: {perfil['missingness']['columns_with_missing']}")

print('\n=== DUPLICADOS Y CLAVES CANDIDATAS ===')
print(f"  Filas duplicadas: {perfil['duplication']['exact_duplicate_rows']} ({perfil['duplication']['duplicate_rate_pct']} %)")
print(f"  Claves candidatas (100% únicas): {perfil['duplication']['unique_key_candidates']}")

print('\n=== GRANULARIDAD TEMPORAL Y ESPACIAL ===')
print('  Temporal:', perfil['granularity']['temporal_features'])
print('  Espacial:', list(perfil['granularity']['spatial_features'].keys()))

In [ ]:
# 4. Tabla de Perfilamiento Detallado Columna por Columna
tabla_resumen = DatasetProfiler.profile_table(df_pluvio)
display(tabla_resumen)

In [ ]:
# 5. Visualizaciones Diagnósticas (Matriz de Ausencias e Histogramas)
fig_missing = DatasetProfiler.plot_missingness_heatmap(df_pluvio)
fig_missing.show()

fig_dist = DatasetProfiler.plot_distributions(df_pluvio, max_features=4)
if fig_dist:
    fig_dist.show()

In [ ]:
# 6. Motor de Imputación Inteligente (Diagnóstico Rubin + Torneo Competitivo)
bridge = NotebookImputerBridge(variance_penalty_weight=1.5)

# Diagnóstico del mecanismo de pérdida
diagnostico = bridge.diagnose(df_pluvio)
print(f'Mecanismo diagnosticado: {diagnostico.diagnosed_mechanism}')
print(f'Estrategia recomendada:  {diagnostico.recommended_strategy}')

# Torneo competitivo de algoritmos con penalización de varianza
df_imputado, tabla_benchmark, resultado = bridge.run_benchmark(df_pluvio)

print('\n=== RESULTADOS DEL BENCHMARK COMPETITIVO ===')
display(tabla_benchmark)
print(f'\nAlgoritmo ganador: {resultado.winning_algorithm_name} (Score: {resultado.winning_score:.4f})')

In [ ]:
# 7. Comparación de Distribución y Preservación de Varianza
numeric_cols = df_pluvio.select_dtypes(include=['number']).columns.tolist()
if numeric_cols:
    col_to_check = numeric_cols[0]
    fig_comp = NotebookImputerBridge.plot_imputation_comparison(
        original_df=df_pluvio,
        imputed_df=df_imputado,
        feature_col=col_to_check
    )
    if fig_comp:
        fig_comp.show()